# ScienceQA — Full Dataset Analysis
**ECE 9660 · Western University**

Coverage of: images, lectures, hints, solutions — broken down by subject, grade, and split.

In [ ]:
import json
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))

plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

PURPLE = '#4F2683'
ORANGE = '#E67E22'
GREEN  = '#27AE60'
BLUE   = '#2980B9'
RED    = '#E74C3C'

SUBJ_COLORS = {
    'natural science':  BLUE,
    'social science':   ORANGE,
    'language science': PURPLE,
}

with open('../results/dataset_analysis.json', encoding='utf-8') as f:
    rows = json.load(f)

df = pd.DataFrame(rows)
print(f'Total examples : {len(df):,}')
print(f'Splits         : {df.split.value_counts().to_dict()}')
df.head(3)

## 1 · Overall Counts per Subject

In [ ]:
SUBJECTS = ['natural science', 'social science', 'language science']
FIELDS   = ['has_image', 'has_lecture', 'has_hint', 'has_solution']
FIELD_LABELS = {
    'has_image':    'Has Image',
    'has_lecture':  'Has Lecture',
    'has_hint':     'Has Hint',
    'has_solution': 'Has Solution',
}

summary = df.groupby('subject')[FIELDS].agg(['sum','mean']).round(3)
summary.columns = [f'{FIELD_LABELS[f]} ({stat})'
                   for f, stat in summary.columns]

# Cleaner table: count + percentage side by side
total_per_subj = df.groupby('subject').size().rename('Total')

rows_out = []
for subj in SUBJECTS:
    sub  = df[df.subject == subj]
    n    = len(sub)
    row  = {'Subject': subj, 'Total': n}
    for field in FIELDS:
        count = sub[field].sum()
        pct   = 100 * count / n
        row[FIELD_LABELS[field]] = f'{int(count):,}  ({pct:.1f}%)'
    rows_out.append(row)

# Total row
row = {'Subject': 'TOTAL', 'Total': len(df)}
for field in FIELDS:
    count = df[field].sum()
    pct   = 100 * count / len(df)
    row[FIELD_LABELS[field]] = f'{int(count):,}  ({pct:.1f}%)'
rows_out.append(row)

tbl = pd.DataFrame(rows_out).set_index('Subject')
print('Full dataset (train + val + test):')
tbl

## 2 · Figure 1 — Field Coverage by Subject (stacked %)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

FIELD_COLORS = {
    'has_image':    RED,
    'has_lecture':  BLUE,
    'has_hint':     GREEN,
    'has_solution': ORANGE,
}

for ax, subj in zip(axes, SUBJECTS):
    sub = df[df.subject == subj]
    n   = len(sub)
    fields = list(FIELD_LABELS.keys())
    pcts   = [100 * sub[f].sum() / n for f in fields]
    colors = [FIELD_COLORS[f] for f in fields]
    labels = [FIELD_LABELS[f] for f in fields]

    bars = ax.barh(labels, pcts, color=colors, alpha=0.85, height=0.55, zorder=3)
    for bar, pct in zip(bars, pcts):
        ax.text(pct + 1, bar.get_y() + bar.get_height()/2,
                f'{pct:.1f}%', va='center', fontsize=9)

    ax.set_xlim(0, 115)
    ax.set_xlabel('Coverage (%)')
    ax.set_title(subj.title(), fontweight='bold',
                 color=SUBJ_COLORS[subj])
    ax.axvline(50, color='gray', linewidth=0.6, linestyle='--', zorder=0)
    ax.grid(axis='x', linestyle='--', alpha=0.3, zorder=0)
    ax.text(0.98, 0.02, f'n={n:,}', transform=ax.transAxes,
            ha='right', va='bottom', fontsize=8, color='gray')

fig.suptitle('Field Coverage by Subject (full dataset)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/ds_fig1_coverage_by_subject.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/ds_fig1_coverage_by_subject.png')

## 3 · Figure 2 — Examples per Grade, coloured by Subject

In [ ]:
GRADE_ORDER = [f'grade{i}' for i in range(1, 13)]
GRADE_LABELS = [f'G{i}' for i in range(1, 13)]

grade_subj = (
    df.groupby(['grade', 'subject'])
    .size()
    .unstack('subject')
    .reindex(GRADE_ORDER)
    .fillna(0)
    .astype(int)
)

fig, ax = plt.subplots(figsize=(11, 4.5))
bottom = np.zeros(len(GRADE_ORDER))
for subj in SUBJECTS:
    if subj not in grade_subj.columns:
        continue
    vals = grade_subj[subj].values
    bars = ax.bar(GRADE_LABELS, vals, bottom=bottom,
                  color=SUBJ_COLORS[subj], label=subj.title(),
                  alpha=0.88, zorder=3)
    # label only tall enough bars
    for bar, v in zip(bars, vals):
        if v > 50:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_y() + bar.get_height()/2,
                    str(v), ha='center', va='center',
                    fontsize=7.5, color='white', fontweight='bold')
    bottom += vals

ax.set_xlabel('Grade')
ax.set_ylabel('Number of examples')
ax.set_title('Dataset Size per Grade — Stacked by Subject', fontweight='bold')
ax.legend(loc='upper right')
ax.grid(axis='y', linestyle='--', alpha=0.3, zorder=0)

plt.tight_layout()
plt.savefig('../results/ds_fig2_grade_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/ds_fig2_grade_distribution.png')

## 4 · Figure 3 — Image vs Text-Only per Grade and Subject

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, subj in zip(axes, SUBJECTS):
    sub = df[df.subject == subj].copy()
    g   = sub.groupby('grade')['has_image'].agg(['sum', 'count']).reindex(GRADE_ORDER).fillna(0)
    g['text_only'] = g['count'] - g['sum']

    x = np.arange(len(GRADE_ORDER))
    ax.bar(x, g['text_only'], label='Text-only', color=BLUE,   alpha=0.85, zorder=3)
    ax.bar(x, g['sum'],       label='Has image', color=RED,    alpha=0.85,
           bottom=g['text_only'], zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels(GRADE_LABELS, fontsize=8)
    ax.set_title(subj.title(), fontweight='bold', color=SUBJ_COLORS[subj])
    ax.set_xlabel('Grade')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.3, zorder=0)

fig.suptitle('Image vs Text-Only per Grade and Subject', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/ds_fig3_image_vs_text_by_grade.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/ds_fig3_image_vs_text_by_grade.png')

## 5 · Figure 4 — Hint Coverage per Grade and Subject

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, subj in zip(axes, SUBJECTS):
    sub = df[df.subject == subj]
    g   = sub.groupby('grade')['has_hint'].mean().reindex(GRADE_ORDER) * 100

    bars = ax.bar(GRADE_LABELS, g.values,
                  color=GREEN, alpha=0.85, zorder=3)
    for bar, v in zip(bars, g.values):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 1,
                    f'{v:.0f}%', ha='center', va='bottom', fontsize=7.5)

    ax.set_ylim(0, 115)
    ax.set_title(subj.title(), fontweight='bold', color=SUBJ_COLORS[subj])
    ax.set_xlabel('Grade')
    ax.set_ylabel('% with hint')
    ax.axhline(50, color='gray', linewidth=0.7, linestyle='--')
    ax.grid(axis='y', linestyle='--', alpha=0.3, zorder=0)
    ax.tick_params(axis='x', labelsize=8)

fig.suptitle('Hint Coverage (%) per Grade and Subject', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/ds_fig4_hint_coverage_by_grade.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/ds_fig4_hint_coverage_by_grade.png')

## 6 · Figure 5 — Lecture Coverage per Grade and Subject

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, subj in zip(axes, SUBJECTS):
    sub = df[df.subject == subj]
    g   = sub.groupby('grade')['has_lecture'].mean().reindex(GRADE_ORDER) * 100

    bars = ax.bar(GRADE_LABELS, g.values,
                  color=BLUE, alpha=0.85, zorder=3)
    for bar, v in zip(bars, g.values):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 1,
                    f'{v:.0f}%', ha='center', va='bottom', fontsize=7.5)

    ax.set_ylim(0, 115)
    ax.set_title(subj.title(), fontweight='bold', color=SUBJ_COLORS[subj])
    ax.set_xlabel('Grade')
    ax.set_ylabel('% with lecture')
    ax.axhline(50, color='gray', linewidth=0.7, linestyle='--')
    ax.grid(axis='y', linestyle='--', alpha=0.3, zorder=0)
    ax.tick_params(axis='x', labelsize=8)

fig.suptitle('Lecture Coverage (%) per Grade and Subject', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/ds_fig5_lecture_coverage_by_grade.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/ds_fig5_lecture_coverage_by_grade.png')

## 7 · Figure 6 — Co-occurrence Heatmap (per subject)
How often does each combination of fields appear together?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
FIELD_SHORT = ['Image', 'Lecture', 'Hint', 'Solution']
FIELD_KEYS  = ['has_image', 'has_lecture', 'has_hint', 'has_solution']

for ax, subj in zip(axes, SUBJECTS):
    sub  = df[df.subject == subj][FIELD_KEYS].astype(float)
    corr = sub.corr()
    corr.index   = FIELD_SHORT
    corr.columns = FIELD_SHORT

    sns.heatmap(corr, annot=True, fmt='.2f', ax=ax,
                cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                linewidths=0.5, cbar=False, square=True)
    ax.set_title(subj.title(), fontweight='bold', color=SUBJ_COLORS[subj])

fig.suptitle('Field Co-occurrence Correlation (per subject)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/ds_fig6_field_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/ds_fig6_field_correlation.png')

## 8 · Number of Choices Distribution

In [ ]:
choice_dist = df.groupby(['subject', 'n_choices']).size().unstack('n_choices').fillna(0).astype(int)
print('Number of answer choices per subject:')
pct = choice_dist.div(choice_dist.sum(axis=1), axis=0).mul(100).round(1)
for subj in SUBJECTS:
    print(f'\n  {subj}:')
    for n_ch in sorted(choice_dist.columns):
        cnt = choice_dist.loc[subj, n_ch] if subj in choice_dist.index else 0
        p   = pct.loc[subj, n_ch] if subj in pct.index else 0
        print(f'    {n_ch} choices: {cnt:>5,}  ({p:.1f}%)')

## 9 · Full Summary Table per Subject × Split

In [ ]:
SPLITS = ['train', 'validation', 'test']

print(f'{"Subject":<22} {"Split":<12} {"Total":>7} {"Image":>8} {"Lecture":>9} {"Hint":>7} {"Solution":>10}')
print('-' * 80)

for subj in SUBJECTS:
    for split in SPLITS:
        sub = df[(df.subject == subj) & (df.split == split)]
        n   = len(sub)
        if n == 0:
            continue
        img  = f"{sub['has_image'].sum():>4} ({100*sub['has_image'].mean():.0f}%)"
        lec  = f"{sub['has_lecture'].sum():>4} ({100*sub['has_lecture'].mean():.0f}%)"
        hint = f"{sub['has_hint'].sum():>4} ({100*sub['has_hint'].mean():.0f}%)"
        sol  = f"{sub['has_solution'].sum():>4} ({100*sub['has_solution'].mean():.0f}%)"
        print(f'{subj:<22} {split:<12} {n:>7,}  {img:>12}  {lec:>12}  {hint:>10}  {sol:>12}')
    print()